# MACE+Graph2Mat

This notebook will show you how to integrate a `MACE` model with `Graph2Mat` through the python API. Note that you can also use `MACE+Graph2Mat` through the Command Line Interface (CLI).

Prerequisites
-------------
Before reading this notebook, **make sure you have read the [notebook on computing a matrix](<./Computing a matrix.ipynb>) and [the notebook on batching](./Batching.ipynb)**, which introduce the basic concepts of `graph2mat` that we are going to assume are already known. Also **we will use exactly the same setup as in the batching notebook**, with the only difference that we will add target matrices to each structure.

In [1]:
import os
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"

In [2]:
import numpy as np
import pandas as pd
import torch

# To load plotly templates for sisl visualization
import sisl.viz

from e3nn import o3

from graph2mat import (
    BasisConfiguration,
    PointBasis,
    BasisTableWithEdges,
    MatrixDataProcessor,
)
from graph2mat.bindings.torch import TorchBasisMatrixDataset, TorchBasisMatrixData

from graph2mat.bindings.e3nn import E3nnGraph2Mat

from graph2mat.tools.viz import plot_basis_matrix


from torch_geometric.loader import DataLoader

/home/saru/anaconda3/envs/basisc/lib/python3.10/site-packages/e3nn/o3/_wigner.py:10: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  _Jd, _W3j_flat, _W3j_indices = torch.load(os.path.join(os.path.dirname(__file__), 'constants.pt'))


Generating a dataset
--------------------

We generate a dataset here just as we have done in the other notebooks.

In [3]:
# The basis
point_1_row = PointBasis("A", R=2, basis="0e", basis_convention="spherical")  # "0e"
point_1_col = PointBasis("A", R=2, basis="2x0e", basis_convention="spherical")
point_2_row = PointBasis("B", R=5, basis="0e + 1o", basis_convention="spherical")
point_2_col = PointBasis("B", R=5, basis="2x0e + 1o", basis_convention="spherical")

this_basis = {'row': [point_1_row,  point_2_row], 'col': [point_1_col, point_2_col]}

# The basis table.
table = BasisTableWithEdges(this_basis)

# The data processor.
processor = MatrixDataProcessor(
    basis_table=table, symmetric_matrix=False,  # Matrix is not square
    sub_point_matrix=False
)

positions = np.array([[0, 0, 0], [6.0, 0, 0], [9.0, 0, 0]])  # positions of the points

config1 = BasisConfiguration(
    point_types=["A", "B", "A"],
    positions=positions,
    basis=this_basis,
    cell=np.eye(3) * 100,
    pbc=(False, False, False),
)

dataset = TorchBasisMatrixDataset([config1], data_processor=processor)


loader = DataLoader(dataset, batch_size=1)

data = next(iter(loader))

In BasisTableWithEdges __init__:
edge_block_shape:  [[1 1 4]
 [2 5 5]]
edge_block_shape_inv:  [[1 4 4]
 [2 2 5]]
concatenated edge_block_shape:  [[1 1 4 4 4]
 [2 5 5 5 2]]


Initializing a MACE model
-------------------------

We will now initialize a normal MACE model.

Note that you must have MACE installed, which you can do with:

```
pip install mace_torch
```

In [4]:
from mace.modules import MACE, RealAgnosticResidualInteractionBlock

num_interactions = 3
hidden_irreps = o3.Irreps("1x0e + 1x1o")

mace_model = MACE(
    r_max=10,
    num_bessel=10,
    num_polynomial_cutoff=10,
    max_ell=2,  # 1,
    interaction_cls=RealAgnosticResidualInteractionBlock,
    interaction_cls_first=RealAgnosticResidualInteractionBlock,
    num_interactions=num_interactions,
    num_elements=2,
    hidden_irreps=hidden_irreps,
    MLP_irreps=o3.Irreps("2x0e"),
    atomic_energies=torch.tensor([0, 0]),
    avg_num_neighbors=2,
    atomic_numbers=[0, 1],
    correlation=2,
    gate=None,
)

cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.


/home/saru/anaconda3/envs/basisc/lib/python3.10/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn(
/home/saru/anaconda3/envs/basisc/lib/python3.10/site-packages/mace/modules/blocks.py:312: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(atomic_energies, dtype=torch.get_default_dtype()),
/home/saru/anaconda3/envs/basisc/lib/python3.10/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.

Now, we can pass our data through the mace model. MACE outputs many things, but we are just interested in the node features, which we can get from the `"node_feats"` key.

In [5]:
mace_output = mace_model(data)
mace_output["node_feats"]

tensor([[ 8.0789e-02,  0.0000e+00,  0.0000e+00,  3.2560e-06, -8.6107e-03,
          0.0000e+00,  0.0000e+00, -9.3593e-04, -8.8840e-04],
        [-3.5286e-01,  0.0000e+00,  0.0000e+00, -7.3308e-04,  4.4133e-01,
          0.0000e+00,  0.0000e+00, -1.0047e-03,  1.5878e-01],
        [ 8.0844e-02,  0.0000e+00,  0.0000e+00, -2.7546e-04, -8.6173e-03,
          0.0000e+00,  0.0000e+00,  2.6839e-03, -8.1314e-04]],
       grad_fn=<CatBackward0>)

Our `Graph2Mat` model will take these node features and convert them to a matrix. Therefore we need to know what its irreps are, and then initialize the `Graph2Mat` module.

In [6]:
# MACE outputs as node features the hidden irreps for each interaction, except
# in the last interaction, where it computes just scalar features.
mace_out_irreps = hidden_irreps * (num_interactions - 1) + str(hidden_irreps[0])

# Initialize the matrix model with this information
matrix_model = E3nnGraph2Mat(
    unique_basis=table,
    irreps=dict(node_feats_irreps=mace_out_irreps),
    symmetric=False,  # Matrix is not square
    # We would need to also implement passing the edge information in order to use
    # preprocessing_edges. As shown later, graph2mat can do this automatically for you.
    preprocessing_edges=None,
)

/home/saru/anaconda3/envs/basisc/lib/python3.10/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn(
/home/saru/anaconda3/envs/basisc/lib/python3.10/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn(
/home/saru/anaconda3/envs/basisc/lib/python3.10/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn(
/

In [7]:
print(matrix_model.summary)

Preprocessing nodes: None
Preprocessing edges: None
Node operations:
 (Ar, Ac)  E3nnSimpleNodeBlock: (1x0e) x (2x0e) -> 2x0e
 (Br, Bc)  E3nnSimpleNodeBlock: (1x0e+1x1o) x (2x0e+1x1o) -> 3x0e+3x1o+1x1e+1x2e
Edge operations:
 (Ar, Ac) E3nnSimpleEdgeBlock: (1x0e) x (2x0e) -> 2x0e.
 (Ar, Bc) E3nnSimpleEdgeBlock: (1x0e) x (2x0e+1x1o) -> 2x0e+1x1o.
 (Br, Ac) E3nnSimpleEdgeBlock: (1x0e+1x1o) x (2x0e) -> 2x0e+2x1o.
 (Br, Bc) E3nnSimpleEdgeBlock: (1x0e+1x1o) x (2x0e+1x1o) -> 3x0e+3x1o+1x1e+1x2e.


Now, we can use the matrix model, passing the node features computed by MACE:

In [8]:
node_labels, edge_labels = matrix_model(data=data, node_feats=mace_output["node_feats"])

In Graph2Mat _get_labels_resort_index: 
BEFORE CALLING get_labels_resorting_array
types:  [0 1 0]
shapes:  [[1 4]
 [2 5]]
transpose_neg:  False
kwargs:  {}
unique_types:  [0 1]
unique_positive_types:  [0 1]
In get_labels_resorting_array:
n_types:  3
ntypes_int:  2
Calculating sizes for type 1: shapes[0, type] = 4, shapes[1, type] = 5
sizes[2] = 4 * 5 = 20
Calculating sizes for type -1: shapes[0, ntypes-type] = 2, shapes[1, n_types - type] = 48
sizes[0] = 2 * 48 = 96
Counting labels for type 0: sizes[1] = 2
type_nlabels[1] = 2
Counting labels for type 1: sizes[2] = 20
type_nlabels[2] = 20
Counting labels for type 0: sizes[1] = 2
type_nlabels[1] = 4
Type 1: offset[prev_type] = 0, type_nlabels[prev_type] = 4
Type -1: offset[-type] = 4, type_nlabels[type] = 20
calculated offset[1] = 4, offset[-1] = 24
sizes:  [96  2 20]
type_nlabels:  [ 0  4 20]
offset:  [24  0  4]
indices.shape:  [24, 0, 0, 0, 0, 0, 0, 0]
AFTER CALLING get_labels_resorting_array
len(indices):  24
indices:  [ 0  1  4  5  6

And plot the obtained matrices:

In [9]:
matrix = processor.matrix_from_data(
    data,
    predictions={"node_labels": node_labels, "edge_labels": edge_labels},
)

plot_basis_matrix(
    matrix[0]*100,
    config1,
    point_lines={"color": "black"},
    basis_lines={"color": "blue"},
    colorscale="temps",
    text=".2f",
    basis_labels=True,
).show()

In sparse.py _blockmatrix_coo_coords:
orbitals_row: [1, 4, 1]
orbitals_col: [2, 5, 2]
edge_index: [[0 1 2 1]
 [1 0 1 2]]
Len rows in _nodes_and_edges_to_coo: 50
Len cols in _nodes_and_edges_to_coo: 50
Shape in _nodes_and_edges_to_coo: (6, 9)
Len node_vals in _nodes_and_edges_to_coo: 24
Len edge_vals in _nodes_and_edges_to_coo: 26
[PointBasis(type='A', R=2, basis=((1, 0, 1),), basis_convention='spherical'), PointBasis(type='B', R=5, basis=((1, 0, 1), (1, 1, -1)), basis_convention='spherical')]


Using MatrixMACE
----------------

If you don't want to handle the details of interacting `MACE` with `Graph2Mat`, you can also use `MatrixMACE`, which takes a mace model and wraps it to also output the `node_labels` and `edge_labels` corresponding to a matrix. 

Internally, it just initializes a `E3nnGraph2Mat` layer. However it can handle the interaction between `MACE` and `Graph2Mat` in more complex cases like having an extra preprocessing step for edges, which needs some extra inputs from MACE.

In [10]:
from graph2mat.models import MatrixMACE
from graph2mat.bindings.e3nn import E3nnEdgeMessageBlock

In [11]:
matrix_mace_model = MatrixMACE(
    mace_model,
    unique_basis=table,
    readout_per_interaction=True,
    edge_hidden_irreps=o3.Irreps("10x0e + 10x1o + 10x2e"),
    symmetric=False,
)

/home/saru/anaconda3/envs/basisc/lib/python3.10/site-packages/torch/jit/_check.py:178: UserWarning:

The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.

/home/saru/anaconda3/envs/basisc/lib/python3.10/site-packages/torch/jit/_check.py:178: UserWarning:

The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.

/home/saru/anaconda3/envs/basisc/lib/python3.10/site-packages/torch/jit/_check.py:178: UserWarning:

The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.

/home/saru/anaconda3/envs/basisc/lib/python3.1

The output of this model is MACE's output plus the `node_labels` and `edge_labels` for the predicted matrix:

In [12]:
print("Data edge index (is used in MatrixMACE forward):")
print(data.edge_index)

Data edge index (is used in MatrixMACE forward):
tensor([[0, 1, 2, 1],
        [1, 0, 1, 2]])


In [13]:
out = matrix_mace_model(data)

out

In Graph2Mat _get_labels_resort_index: 
BEFORE CALLING get_labels_resorting_array
types:  [0 1 0]
shapes:  [[1 4]
 [2 5]]
transpose_neg:  False
kwargs:  {}
unique_types:  [0 1]
unique_positive_types:  [0 1]
In get_labels_resorting_array:
n_types:  3
ntypes_int:  2
Calculating sizes for type 1: shapes[0, type] = 4, shapes[1, type] = 5
sizes[2] = 4 * 5 = 20
Calculating sizes for type -1: shapes[0, ntypes-type] = 2, shapes[1, n_types - type] = 48
sizes[0] = 2 * 48 = 96
Counting labels for type 0: sizes[1] = 2
type_nlabels[1] = 2
Counting labels for type 1: sizes[2] = 20
type_nlabels[2] = 20
Counting labels for type 0: sizes[1] = 2
type_nlabels[1] = 4
Type 1: offset[prev_type] = 0, type_nlabels[prev_type] = 4
Type -1: offset[-type] = 4, type_nlabels[type] = 20
calculated offset[1] = 4, offset[-1] = 24
sizes:  [96  2 20]
type_nlabels:  [ 0  4 20]
offset:  [24  0  4]
indices.shape:  [24, 0, 0, 0, 0, 0, 0, 0]
AFTER CALLING get_labels_resorting_array
len(indices):  24
indices:  [ 0  1  4  5  6

{'energy': tensor([0.3346], grad_fn=<SumBackward1>),
 'node_energy': tensor([0.0140, 0.3066, 0.0140], grad_fn=<SumBackward1>),
 'contributions': tensor([[ 0.0000e+00,  0.0000e+00, -5.1618e-02,  3.8588e-01,  3.1516e-04]],
        grad_fn=<StackBackward0>),
 'forces': None,
 'edge_forces': None,
 'virials': None,
 'stress': None,
 'atomic_virials': None,
 'atomic_stresses': None,
 'displacement': tensor([[[0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.]]]),
 'hessian': None,
 'node_feats': tensor([[ 8.0789e-02,  0.0000e+00,  0.0000e+00,  3.2560e-06, -8.6107e-03,
           0.0000e+00,  0.0000e+00, -9.3593e-04, -8.8840e-04],
         [-3.5286e-01,  0.0000e+00,  0.0000e+00, -7.3308e-04,  4.4133e-01,
           0.0000e+00,  0.0000e+00, -1.0047e-03,  1.5878e-01],
         [ 8.0844e-02,  0.0000e+00,  0.0000e+00, -2.7546e-04, -8.6173e-03,
           0.0000e+00,  0.0000e+00,  2.6839e-03, -8.1314e-04]],
        grad_fn=<CatBackward0>),
 'node_labels': tensor([-5.0492e-04, -5.1480e-04

You can of course plot the predicted matrices:

In [14]:
matrix = processor.matrix_from_data(data, predictions=out)
from matplotlib import pyplot as plt
plot_basis_matrix(
    matrix[0]*1e6,
    config1,
    point_lines={"color": "black"},
    basis_lines={"color": "blue"},
    colorscale="temps",
    text=".2f",
    basis_labels=True,
).show()


In sparse.py _blockmatrix_coo_coords:
orbitals_row: [1, 4, 1]
orbitals_col: [2, 5, 2]
edge_index: [[0 1 2 1]
 [1 0 1 2]]
Len rows in _nodes_and_edges_to_coo: 50
Len cols in _nodes_and_edges_to_coo: 50
Shape in _nodes_and_edges_to_coo: (6, 9)
Len node_vals in _nodes_and_edges_to_coo: 24
Len edge_vals in _nodes_and_edges_to_coo: 26
[PointBasis(type='A', R=2, basis=((1, 0, 1),), basis_convention='spherical'), PointBasis(type='B', R=5, basis=((1, 0, 1), (1, 1, -1)), basis_convention='spherical')]


In [15]:
print("out edge labels  - obtained from the matrix MACE model and data")
print('Number of edge labels:', len(out['edge_labels']))
print(out['edge_labels']*1e8)

out edge labels  - obtained from the matrix MACE model and data
Number of edge labels: 26
tensor([  110.5833,    63.6270,     0.0000,     0.0000,     8.7536,  -100.4088,
          -85.1992,     0.0000,     0.0000,     0.0000,     0.0000,    61.3142,
          -43.7274, -1169.8820,  -711.9777,     0.0000,     0.0000,   377.3493,
           32.7543,   403.0532,     0.0000,     0.0000,     0.0000,     0.0000,
         1294.3242,  1155.5604], grad_fn=<MulBackward0>)


# Rotating matrix

A matrix should rotate equivariantly if we rotate the configuration given to mace via the Data. Let us try!

Thing sthat remain constant under rotation: the basis we defined and the thing sthat depend on it.

- table object: an processed object with the basis of each point basis (and the basis points point_1, point_2, point_3, point_4)
- data_processor object processor : has info of how to process the basis, i.e., information about the edges and pointers.
- mace_model and matrix_mace_model : it is just the architecture of the mace model, so we use the same model for both the original and rotated configurations. Matrixmace is just the matrixed version of the mace model, so it is also the same for both configurations.

In [16]:
positions_rot = np.array([[0, 0, 0], [0, 6.0, 0], [0, 9.0, 0]])  # rotated positions

config1_rot = BasisConfiguration(
    point_types=["A", "B", "A"],
    positions=positions_rot,  # changed positions to rotated ones
    basis=this_basis,
    cell=np.eye(3) * 100,
    pbc=(False, False, False),
)


dataset_rot = TorchBasisMatrixDataset([config1_rot], data_processor=processor)


loader_rot = DataLoader(dataset_rot, batch_size=1)

data_rot = next(iter(loader_rot))

In [17]:
out_rot = matrix_mace_model(data_rot)

In Graph2Mat _get_labels_resort_index: 
BEFORE CALLING get_labels_resorting_array
types:  [0 1 0]
shapes:  [[1 4]
 [2 5]]
transpose_neg:  False
kwargs:  {}
unique_types:  [0 1]
unique_positive_types:  [0 1]
In get_labels_resorting_array:
n_types:  3
ntypes_int:  2
Calculating sizes for type 1: shapes[0, type] = 4, shapes[1, type] = 5
sizes[2] = 4 * 5 = 20
Calculating sizes for type -1: shapes[0, ntypes-type] = 2, shapes[1, n_types - type] = 48
sizes[0] = 2 * 48 = 96
Counting labels for type 0: sizes[1] = 2
type_nlabels[1] = 2
Counting labels for type 1: sizes[2] = 20
type_nlabels[2] = 20
Counting labels for type 0: sizes[1] = 2
type_nlabels[1] = 4
Type 1: offset[prev_type] = 0, type_nlabels[prev_type] = 4
Type -1: offset[-type] = 4, type_nlabels[type] = 20
calculated offset[1] = 4, offset[-1] = 24
sizes:  [96  2 20]
type_nlabels:  [ 0  4 20]
offset:  [24  0  4]
indices.shape:  [24, 0, 0, 0, 0, 0, 0, 0]
AFTER CALLING get_labels_resorting_array
len(indices):  24
indices:  [ 0  1  4  5  6

You can of course plot the predicted matrices:

In [18]:
matrix_rot = processor.matrix_from_data(data_rot, predictions=out_rot)

plot_basis_matrix(
    matrix_rot[0]*1e6,
    config1_rot,
    point_lines={"color": "black"},
    basis_lines={"color": "blue"},
    colorscale="temps",
    text=".2f",
    basis_labels=True,
).show()

In sparse.py _blockmatrix_coo_coords:
orbitals_row: [1, 4, 1]
orbitals_col: [2, 5, 2]
edge_index: [[0 1 2 1]
 [1 0 1 2]]
Len rows in _nodes_and_edges_to_coo: 50
Len cols in _nodes_and_edges_to_coo: 50
Shape in _nodes_and_edges_to_coo: (6, 9)
Len node_vals in _nodes_and_edges_to_coo: 24
Len edge_vals in _nodes_and_edges_to_coo: 26
[PointBasis(type='A', R=2, basis=((1, 0, 1),), basis_convention='spherical'), PointBasis(type='B', R=5, basis=((1, 0, 1), (1, 1, -1)), basis_convention='spherical')]


In [19]:
print("out_rot edge labels  - obtained from the matrix MACE model and data")
print('Number of edge labels:', len(out_rot['edge_labels']))
print(out_rot['edge_labels']*1e8)

out_rot edge labels  - obtained from the matrix MACE model and data
Number of edge labels: 26
tensor([  110.5833,    63.6270,     8.7536,     0.0000,     0.0000,  -100.4088,
          -85.1992,    61.3142,   -43.7274,     0.0000,     0.0000,     0.0000,
            0.0000, -1169.8820,  -711.9777,   377.3493,     0.0000,     0.0000,
           32.7543,   403.0532,  1294.3242,  1155.5604,     0.0000,     0.0000,
            0.0000,     0.0000], grad_fn=<MulBackward0>)


Summary and next steps
----------------------

In this notebook we learned **how to interface MACE with Graph2Mat**.

The **next steps** could be:

- **Train a MACE+Graph2Mat model** following the steps in [this notebook](<./Fitting matrices.ipynb>), replacing the model by the `MACE+Graph2Mat` model.